In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/ecb_policy_rate_raw.csv")

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

print()
print(df.head())

print()
print(df.tail())

Shape: (64, 40)
Columns:
['KEY', 'FREQ', 'REF_AREA', 'CURRENCY', 'PROVIDER_FM', 'INSTRUMENT_FM', 'PROVIDER_FM_ID', 'DATA_TYPE_FM', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS', 'OBS_CONF', 'OBS_PRE_BREAK', 'OBS_COM', 'TIME_FORMAT', 'BREAKS', 'COLLECTION', 'COMPILING_ORG', 'DISS_ORG', 'DOM_SER_IDS', 'FM_CONTRACT_TIME', 'FM_COUPON_RATE', 'FM_IDENTIFIER', 'FM_LOT_SIZE', 'FM_MATURITY', 'FM_OUTS_AMOUNT', 'FM_PUT_CALL', 'FM_STRIKE_PRICE', 'PUBL_MU', 'PUBL_PUBLIC', 'UNIT_INDEX_BASE', 'COMPILATION', 'COVERAGE', 'DECIMALS', 'SOURCE_AGENCY', 'SOURCE_PUB', 'TITLE', 'TITLE_COMPL', 'UNIT', 'UNIT_MULT']

                         KEY FREQ REF_AREA CURRENCY PROVIDER_FM INSTRUMENT_FM  \
0  FM.B.U2.EUR.4F.KR.DFR.CHG    B       U2      EUR          4F            KR   
1  FM.B.U2.EUR.4F.KR.DFR.CHG    B       U2      EUR          4F            KR   
2  FM.B.U2.EUR.4F.KR.DFR.CHG    B       U2      EUR          4F            KR   
3  FM.B.U2.EUR.4F.KR.DFR.CHG    B       U2      EUR          4F            KR   
4

In [ ]:
ecb = df[
    ["TIME_PERIOD", "OBS_VALUE"]
].copy()

ecb["TIME_PERIOD"] = pd.to_datetime(ecb["TIME_PERIOD"])
ecb["OBS_VALUE"] = pd.to_numeric(ecb["OBS_VALUE"])

ecb = ecb.sort_values("TIME_PERIOD").reset_index(drop=True)

print(ecb.to_string(index=False))

TIME_PERIOD  OBS_VALUE
 1999-01-04       0.75
 1999-01-22      -0.75
 1999-04-09      -0.50
 1999-11-05       0.50
 2000-02-04       0.25
 2000-03-17       0.25
 2000-04-28       0.25
 2000-06-09       0.50
 2000-09-01       0.25
 2000-10-06       0.25
 2001-05-11      -0.25
 2001-08-31      -0.25
 2001-09-18      -0.50
 2001-11-09      -0.50
 2002-12-06      -0.50
 2003-03-07      -0.25
 2003-06-06      -0.50
 2005-12-06       0.25
 2006-03-08       0.25
 2006-06-15       0.25
 2006-08-09       0.25
 2006-10-11       0.25
 2006-12-13       0.25
 2007-03-14       0.25
 2007-06-13       0.25
 2008-07-09       0.25
 2008-10-08      -0.50
 2008-10-09       0.50
 2008-11-12      -0.50
 2008-12-10      -0.75
 2009-01-21      -1.00
 2009-03-11      -0.50
 2009-04-08      -0.25
 2011-04-13       0.25
 2011-07-13       0.25
 2011-11-09      -0.25
 2011-12-14      -0.25
 2012-07-11      -0.25
 2013-05-08       0.00
 2013-11-13       0.00
 2014-06-11      -0.10
 2014-09-10      -0.10
 2015-12-09

In [3]:
print("DATA TYPE:")
print(df["DATA_TYPE_FM"].unique())

print()
print("TITLE:")
print(df["TITLE"].iloc[0])

print()
print("Latest changes:")
print(ecb.tail(10).to_string(index=False))

DATA TYPE:
<StringArray>
['CHG']
Length: 1, dtype: str

TITLE:
Deposit facility - date of changes (raw data) - Change in percentage points compared to previous rate

Latest changes:
TIME_PERIOD  OBS_VALUE
 2023-09-20       0.25
 2024-06-12      -0.25
 2024-09-18      -0.25
 2024-10-23      -0.25
 2024-12-18      -0.25
 2025-02-05      -0.25
 2025-03-12      -0.25
 2025-04-23      -0.25
 2025-06-11      -0.25
 2026-06-17       0.25


In [4]:
print(ecb.head(15).to_string(index=False))

TIME_PERIOD  OBS_VALUE
 1999-01-04       0.75
 1999-01-22      -0.75
 1999-04-09      -0.50
 1999-11-05       0.50
 2000-02-04       0.25
 2000-03-17       0.25
 2000-04-28       0.25
 2000-06-09       0.50
 2000-09-01       0.25
 2000-10-06       0.25
 2001-05-11      -0.25
 2001-08-31      -0.25
 2001-09-18      -0.50
 2001-11-09      -0.50
 2002-12-06      -0.50


1. Reconstruct the policy-rate level

In [5]:
ecb["TIME_PERIOD"] = pd.to_datetime(ecb["TIME_PERIOD"])

ecb = ecb.sort_values("TIME_PERIOD").reset_index(drop=True)

# Initial ECB deposit facility rate at the start of Stage Three
initial_rate = 2.00

ecb["ecb_deposit_facility_rate"] = (
    initial_rate + ecb["OBS_VALUE"].cumsum()
)

2. Create the final processed dataset

In [6]:
ecb_policy_rate = ecb[
    [
        "TIME_PERIOD",
        "OBS_VALUE",
        "ecb_deposit_facility_rate"
    ]
].copy()

ecb_policy_rate = ecb_policy_rate.rename(columns={
    "TIME_PERIOD": "date",
    "OBS_VALUE": "rate_change"
})

ecb_policy_rate["date"] = pd.to_datetime(ecb_policy_rate["date"])

ecb_policy_rate = ecb_policy_rate.sort_values(
    "date"
).reset_index(drop=True)

Validate 

In [7]:
print("Shape:", ecb_policy_rate.shape)
print("Missing values:")
print(ecb_policy_rate.isna().sum())

print("Duplicate dates:",
      ecb_policy_rate["date"].duplicated().sum())

print("Date range:",
      ecb_policy_rate["date"].min(),
      "to",
      ecb_policy_rate["date"].max())

print(ecb_policy_rate.head())
print(ecb_policy_rate.tail())

Shape: (64, 3)
Missing values:
date                         0
rate_change                  0
ecb_deposit_facility_rate    0
dtype: int64
Duplicate dates: 0
Date range: 1999-01-04 00:00:00 to 2026-06-17 00:00:00
        date  rate_change  ecb_deposit_facility_rate
0 1999-01-04         0.75                       2.75
1 1999-01-22        -0.75                       2.00
2 1999-04-09        -0.50                       1.50
3 1999-11-05         0.50                       2.00
4 2000-02-04         0.25                       2.25
         date  rate_change  ecb_deposit_facility_rate
59 2025-02-05        -0.25                       2.75
60 2025-03-12        -0.25                       2.50
61 2025-04-23        -0.25                       2.25
62 2025-06-11        -0.25                       2.00
63 2026-06-17         0.25                       2.25


Save it!

In [9]:
output_path = "../data/processed/ecb_policy_rate.csv"

ecb_policy_rate.to_csv(
    output_path,
    index=False
)

print("Processed ECB policy rate saved to:", output_path)

Processed ECB policy rate saved to: ../data/processed/ecb_policy_rate.csv
